# 02 – Feature Engineering

## From Raw Prices to Predictive Features

Good feature engineering for electricity price forecasting requires strict discipline:
**no future information can be used as an input at prediction time.**

We apply the following feature groups:
1. **Calendar** features (hour, weekday, month, holiday flag)
2. **Cyclical encodings** (sin/cos transforms to preserve periodicity)
3. **Lag features** (price 24h, 48h, 168h ago — strictly in the past)
4. **Rolling statistics** (mean/std computed with a 1-hour shift to avoid leakage)
5. **Weather features** (temperature + lag + anomaly from seasonal mean)

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.features import build_feature_matrix, FEATURE_COLS, TARGET_COL

df_raw = pd.read_parquet('../data/processed/hourly_market.parquet')
df = build_feature_matrix(df_raw)

print(f"Raw rows: {len(df_raw):,}  →  After feature build: {len(df):,}")
print(f"Features: {len([c for c in FEATURE_COLS if c in df.columns])}")
print("\nFeature columns:")
for c in FEATURE_COLS:
    if c in df.columns:
        print(f"  {c}")


Raw rows: 52,608  →  After feature build: 51,888
Features: 30

Feature columns:
  hour
  day_of_week
  month
  year
  quarter
  is_weekend
  is_holiday
  season
  hour_sin
  hour_cos
  month_sin
  month_cos
  dow_sin
  dow_cos
  price_lag_24
  price_lag_48
  price_lag_168
  price_roll_mean_24
  price_roll_std_24
  price_roll_mean_168
  price_roll_std_168
  temp_istanbul
  temp_istanbul_lag24
  temp_istanbul_dev
  temp_ankara
  temp_ankara_lag24
  temp_ankara_dev
  temp_izmir
  temp_izmir_lag24
  temp_izmir_dev


## 1. Lag Feature Correlation

In [2]:
lags = ['price_lag_24', 'price_lag_48', 'price_lag_168']
corrs = {lag: df[TARGET_COL].corr(df[lag]) for lag in lags}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lag in zip(axes, lags):
    sample = df.sample(2000, random_state=42)
    ax.scatter(sample[lag], sample[TARGET_COL], alpha=0.15, s=3, color='#1565C0')
    ax.set_xlabel(f'{lag} (TL/MWh)')
    ax.set_ylabel('Actual price (TL/MWh)')
    ax.set_title(f'{lag}\nr = {corrs[lag]:.3f}')
    ax.grid(True, alpha=0.3)
plt.suptitle('Lag Feature vs Target Correlation', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/02_lag_correlations.png', dpi=150)
plt.show()
print("Correlations:", corrs)


Correlations: {'price_lag_24': 0.9894253910965939, 'price_lag_48': 0.9788722548008622, 'price_lag_168': 0.9999798144214668}


## 2. Cyclical Encoding: sin/cos vs raw hour

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hours = np.arange(24)
axes[0].plot(hours, hours, 'o-', color='#E53935')
axes[0].set_title('Raw hour encoding\n(hour 23 and 0 are far apart!)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Encoded value')
axes[0].grid(True, alpha=0.3)

sin_vals = np.sin(2 * np.pi * hours / 24)
cos_vals = np.cos(2 * np.pi * hours / 24)
axes[1].plot(sin_vals, cos_vals, 'o-', color='#1565C0')
for h in [0, 6, 12, 18]:
    axes[1].annotate(f'h={h}', (sin_vals[h], cos_vals[h]), textcoords='offset points', xytext=(5,5))
axes[1].set_title('Cyclical encoding (sin, cos)\n(hour 23 and 0 are neighbours)')
axes[1].set_xlabel('sin(hour)')
axes[1].set_ylabel('cos(hour)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/02_cyclical_encoding.png', dpi=150)
plt.show()


## 3. Feature Correlation Heatmap (top features vs target)

In [4]:
feature_cols_present = [c for c in FEATURE_COLS if c in df.columns]
corr_with_target = df[feature_cols_present + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
top_features = corr_with_target.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1565C0' if v > 0 else '#E53935' for v in top_features.values]
ax.barh(top_features.index[::-1], top_features.values[::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with target price')
ax.set_title('Top 15 Features by Absolute Correlation with Target', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/02_feature_correlations.png', dpi=150)
plt.show()
print("Top 5 features:")
print(top_features.head())


Top 5 features:
price_lag_168          0.999980
price_roll_mean_24     0.991273
price_lag_24           0.989425
price_roll_mean_168    0.987435
price_roll_std_168     0.981820
Name: dam_price_try_mwh, dtype: float64


## Key Design Decisions

- **t-168 lag** is the most informative single feature (same hour last week)
- **Cyclical encodings** prevent the model from treating 23:00 and 00:00 as distant
- **Rolling stats** use `.shift(1)` before `.rolling()` — this is the critical anti-leakage step
- Weather deviation from seasonal mean captures anomaly signal (heatwaves, cold snaps)